# 00 - Environment Validation

## Objetivo
Confirmar que o ambiente de execução tem as versões e dependências
esperadas antes de rodar qualquer notebook de demonstração, e executar um
smoke test mínimo do `ibqn` sobre o SeQUeNCe real.

## Conceitos
- O pacote `sequence` (SeQUeNCe) é vendorizado como submódulo Git em
  `SeQUeNCe/`, fixado em um commit específico (ver `docs/sequence_code_analysis.md`).
- O pacote `ibqn` (`src/ibqn/`) nunca modifica o núcleo do SeQUeNCe -
  apenas o adapta via composição (`network.sequence_adapter`,
  `execution.sequence_executor`).
- Este notebook não instala nada: instalação de dependências é
  responsabilidade do ambiente (`pyproject.toml` + `.venv`), nunca de uma
  célula de notebook.


## Configuração


In [1]:
from ibqn.demos.environment import collect_environment_info, run_smoke_test

info = collect_environment_info()
info


EnvironmentInfo(python_version='3.13.14', platform='Windows-11-10.0.26200-SP0', sequence_version='1.0.0', sequence_commit='1f2680a5b9065e708a7497adc53a95b108029b98', ibqn_version='0.1.0', project_commit='1014042cc69516ed89107a6825643c5e54729ac0', default_formalism='bell_diagonal', dependency_versions={'pydantic': '2.13.4', 'yaml': '6.0.3', 'networkx': '3.6.1', 'numpy': '2.5.1', 'scipy': '1.18.0', 'matplotlib': '3.11.0', 'pandas': '3.0.3', 'qutip': '5.3.0', 'stim': '1.16.0', 'gmpy2': '2.3.1', 'nbformat': '5.10.4', 'nbclient': '0.11.0'})

## Execução

Smoke test: constrói a menor topologia possível (2 roteadores + 1 nó BSM) e roda a `Timeline` até o fim.


In [2]:
smoke_result = run_smoke_test()
smoke_result


2026-10-02 15:03:07,602 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - topology built: 2 routers, 1 quantum links, formalism=bell_diagonal, seed=0


2026-10-02 15:03:07,608 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - running simulation, stop_time=0.010s


{'routers_built': 2, 'graph_nodes': 2, 'graph_edges': 1, 'events_executed': 0}

## Resultados


In [3]:
import pandas as pd

rows = [
    {"field": "Python version", "value": info.python_version},
    {"field": "Platform", "value": info.platform},
    {"field": "SeQUeNCe version", "value": info.sequence_version},
    {"field": "SeQUeNCe submodule commit", "value": info.sequence_commit or "unavailable (not a git checkout)"},
    {"field": "ibqn version", "value": info.ibqn_version},
    {"field": "Default quantum formalism", "value": info.default_formalism},
]
pd.DataFrame(rows)


,field,value
0,Python version,3.13.14
1,Platform,Windows-11-10.0.26200-SP0
2,SeQUeNCe version,1.0.0
3,SeQUeNCe submodule commit,1f2680a5b9065e708a7497adc53a95b108029b98
4,ibqn version,0.1.0
5,Default quantum formalism,bell_diagonal


In [4]:
dependencies_df = pd.DataFrame(
    [{"package": name, "version": version} for name, version in info.dependency_versions.items()]
)
dependencies_df


,package,version
0,pydantic,2.13.4
1,yaml,6.0.3
2,networkx,3.6.1
3,numpy,2.5.1
4,scipy,1.18.0
5,matplotlib,3.11.0
6,pandas,3.0.3
7,qutip,5.3.0
8,stim,1.16.0
9,gmpy2,2.3.1


In [5]:
assert not (dependencies_df["version"] == "NOT INSTALLED").any(), "one or more required dependencies are missing"
assert smoke_result["routers_built"] == 2
assert smoke_result["graph_edges"] == 1
print("All environment checks passed.")


All environment checks passed.


## Interpretação
Todas as dependências listadas acima foram importadas com sucesso a partir
do ambiente atual (`dependency_versions` reflete `importlib.import_module`
real, nunca uma lista fixa). O smoke test confirma que o `SequenceAdapter`
consegue construir uma topologia real do SeQUeNCe (via `RouterNetTopo`) e
rodar a `Timeline` até `stop_time` sem exceções.

## Limitações
- `sequence_commit` só é preenchido quando o pacote `sequence` importado
  vem de um checkout Git (é o caso neste projeto, via submódulo); uma
  instalação via `pip install sequence` não teria commit disponível.
- O smoke test não faz nenhuma requisição de entrelaçamento (não há
  `EntanglementIntent` envolvido) - ele valida apenas que a construção e a
  execução básicas do simulador funcionam. Os notebooks seguintes (`01`
  em diante) validam a física real.

## Conclusão
Ambiente validado: versões, dependências e um smoke test real do SeQUeNCe
via `ibqn.network.sequence_adapter` confirmados nesta execução.
